In [ ]:
import openmc 
from openmc.deplete import d1s
from openmc.data import effective_dose
from openmc_fusion_benchmarks.neutron_sources import fng_source

model = openmc.Model()

# Define the materials
material = openmc.Materials.from_xml('materials.xml')
# material.cross_sections =  '/home/grego01/Nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'
model.materials = material

# Define the geometry
geom = openmc.Geometry.from_xml('geometry.xml')
model.geometry = geom

# Define Settings
# source definition
fng_center = (0, 0, 0)
fng_uvw = (0., 1., 0)

source = fng_source(center=fng_center,
                    reference_uvw=fng_uvw, beam_energy=230)

# settings
settings = openmc.Settings(run_mode='fixed source')
settings.batches = 100
settings.particles = 1000
settings.source = source
settings.photon_transport = True
settings.verbosity = 5
settings.output = {'tallies': False}

model.settings = settings

############################################################################
# Specify Tallies
model.tallies = openmc.Tallies()

# particle filters
neutron_filter = openmc.ParticleFilter(['neutron'])
photon_filter = openmc.ParticleFilter(['photon'])

# cell filters
cell_filter_9 = openmc.CellFilter([621, 622, 624, 625])
cell_filter_10 = openmc.CellFilter([178, 179, 180, 181, 182, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 636, 637, 638])
cell_filter_photon = openmc.CellFilter([227])

# Energy filters
energies, coeffs = effective_dose.dose.dose_coefficients('photon', data_source='icrp74')
dose_filter = openmc.EnergyFunctionFilter(energies, coeffs)

tally = openmc.Tally(name="sdr_dose")
tally.filters = [cell_filter_photon, photon_filter, dose_filter]
tally.scores = ['flux']
model.tallies.append(tally)

tally = openmc.Tally(name="foil_activation")
tally.filters = [cell_filter_10, neutron_filter]
tally.scores = ['(n,gamma)', '(n,p)', '(n,2n)', '(n,3n)']
model.tallies.append(tally)

tally = openmc.Tally(name="neutron_flux")
tally.filters = [cell_filter_photon, neutron_filter]
tally.scores = ['flux']
model.tallies.append(tally)

model.export_to_xml()

# openmc.config['chain_file'] = r'/home/davide/openmc_models/CROSS_SECTIONS/decay/endfb-viii.0-chain/chain_endfb80_sfr.xml'
# model.settings.photon_transport = True
# model.settings.use_decay_photons = True
# d1s.prepare_tallies(model)

/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=4.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=5.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material in

['Be10',
 'In122',
 'Mo91',
 'P32',
 'Co60',
 'Cd119',
 'P34',
 'Fe59',
 'P36',
 'Sn111',
 'Nb94_m1',
 'Mn52',
 'Sc47',
 'S37',
 'Sn121',
 'Pb202',
 'N13',
 'Sn110',
 'Mn53',
 'P33',
 'Tl204',
 'Nb97',
 'Co58',
 'Hg203',
 'V49',
 'Cd113',
 'Cr55',
 'Cu66',
 'S35',
 'Fe61',
 'Be11',
 'O15',
 'Pb207_m1',
 'In118',
 'Nb96',
 'Nb91',
 'B12',
 'Sn123',
 'Nb94',
 'Co62',
 'Si33',
 'Tl206',
 'Cd109',
 'Ni59',
 'Cd115',
 'Pb205',
 'Co57',
 'Ni57',
 'Pb203',
 'In119',
 'Cr49',
 'V50',
 'C14',
 'Sn125_m1',
 'In124',
 'Mn56',
 'Sn113_m1',
 'Al28',
 'Ni63',
 'In116',
 'In117',
 'Mo93',
 'Si31',
 'Cu62',
 'Cd117',
 'Co61',
 'Cd121',
 'Mn57',
 'Fe55',
 'Pb204',
 'V53',
 'Sn121_m1',
 'Si27',
 'Tl207',
 'Cr51',
 'Ni65',
 'Pb209',
 'Co64',
 'Sc48',
 'In120',
 'Nb100',
 'Mo99',
 'Mo93_m1',
 'In115',
 'Hg205',
 'In112',
 'Co60_m1',
 'Li8',
 'Sn113',
 'S31',
 'Mn54',
 'Zr95',
 'Sn123_m1',
 'In114',
 'Cu64',
 'Nb98',
 'V54',
 'Ti51',
 'Zr89',
 'P30',
 'Sn117_m1',
 'Cd116',
 'Mn58',
 'Fe53',
 'Nb95',
 'Zr97

In [3]:
sp = openmc.StatePoint('statepoint.100.h5')

# only for d1s
# d1s depletion
model.settings.use_decay_photons = True

nuclides = d1s.prepare_tallies(model)

time_steps = [19440, 61680, 32940, 54840, 15720, 6360, 8940]
source_rates = [2.32e10, 0.0, 2.87e10, 0.0, 1.90e10, 0.0, 1.36e10]

factors = d1s.time_correction_factors(nuclides, time_steps, source_rates)
tally_dose = sp.get_tally(name='sdr_dose')
tally_rr = sp.get_tally(name='foil_activation')
tally_neutron = sp.get_tally(name='neutron_flux')

print(tally_neutron.get_values())
tally_dose = d1s.apply_time_correction(tally_dose, factors)
print(tally_dose.get_values())
print(tally_rr.get_values())

[[[0.26445368]]]
[[[1.03990438e+08]]]
[[[3.67502023e-05 6.46903269e-06 1.67748943e-05 2.63816649e-11]]

 [[1.13221492e-04 1.79152332e-05 3.45488454e-05 4.81783820e-11]]

 [[1.40904579e-04 1.83341352e-05 3.38008251e-05 1.83798004e-11]]

 [[2.41817411e-04 2.53086680e-05 4.54728040e-05 4.39555154e-11]]

 [[2.62926196e-04 3.68301380e-05 7.38184346e-05 4.31327194e-11]]

 [[1.27507746e-04 2.29772122e-05 4.44916801e-05 3.81074201e-11]]

 [[2.10604169e-04 3.88947867e-05 8.33557906e-05 6.02340022e-11]]

 [[3.12915450e-04 4.93676629e-05 1.00941841e-04 1.08210012e-10]]

 [[4.13450953e-04 6.23568711e-05 1.21567713e-04 5.79078992e-11]]

 [[8.74044515e-04 1.18953211e-04 2.26836008e-04 1.14832786e-10]]

 [[1.00080922e-03 1.03083182e-04 1.96130396e-04 1.01625706e-10]]

 [[1.27189088e-03 1.05260527e-04 2.00898027e-04 6.12027046e-11]]

 [[6.50057174e-04 4.71663304e-05 7.87969084e-05 9.31801174e-12]]

 [[3.70258747e-04 3.18464918e-05 5.11795452e-05 2.09876339e-12]]

 [[3.38179053e-05 7.40629410e-06 1.743